In [ ]:
import itertools
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dataset_root = pathlib.Path('/home/cyril/development/data/2nd StepUP Competition Datasets')

datapath_train = dataset_root / '1 - Training'
datapath_ref = dataset_root / '2 - Reference' 
datapath_probe = dataset_root / '3 - Probe'

for filepath in [datapath_train, datapath_ref, datapath_probe]:
  assert(filepath.exists())

In [ ]:
participant_IDs = np.arange(1,151) 
speed_IDs = ['W1','W2','W3','W4']
footwear_IDs = ['BF','ST','P1','P2']

metadata_lst = []
for participant_ID, footwear_ID, speed_ID in itertools.product(participant_IDs, footwear_IDs, speed_IDs):
  metadata_path = datapath_train / f'{participant_ID:03}' / footwear_ID / speed_ID / 'metadata.csv'
  metadata_lst.append(pd.read_csv(metadata_path))

metadata_train = pd.concat(metadata_lst).reset_index(drop=True)

# optional: remove any samples flagged to 'Exclude' - these include incomplete or outlier footsteps 
metadata_train = metadata_train.query('Exclude == False')

metadata_train.head()

In [ ]:
label_map_fw = {
    "BF": 0,
    "ST": 1,
    "P1": 2,
    "P2": 3
}
label_map_sp = {
    "W1": 0,
    "W2": 1,
    "W3": 2,
    "W4": 3
}

## Export 3 : same as 1 (log norm) but index dictionary

In [ ]:
X = np.memmap('X_AllStepsR_f32_norm.memmap', dtype=np.float32, mode="w+", shape=(len(metadata_train), 101, 75, 40))
Y = np.zeros((len(metadata_train), 3), dtype=np.uint8)


# index[pid][shoe][speed] = [sample indices]
index = {}

i = 0
flipped = 0

for row in metadata_train.itertuples(index=True):
    # not clean as we reload npz each time, but only run once...
    sample_path = datapath_train / f'{row.ParticipantID:03}' / row.Footwear / row.Speed / 'pipeline_1.npz'
    sample = np.load(sample_path)[f'{str(row.FootstepID)}'].astype(np.float32)

    # todo: if left do a flip
    if row.Side == 'Left':
        flipped += 1
        sample = sample[:, :, ::-1]

    pid   = row.ParticipantID
    shoe  = label_map_fw[row.Footwear]
    speed = label_map_sp[row.Speed]
    
    if pid not in index:
        index[pid] = {}
    if shoe not in index[pid]:
        index[pid][shoe] = {}
    if speed not in index[pid][shoe]:
        index[pid][shoe][speed] = []

    index[pid][shoe][speed].append(i)
    X[i] = np.log1p(sample) / 7.0
    Y[i] = [pid, shoe, speed]
    
    i += 1

X.flush()
np.save('Y_AllStepsR.npy', Y)
np.save("XId_AllStepsR.npy", dict(index), allow_pickle=True)

In [ ]:
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(len(metadata_train), 101, 75, 40))

In [ ]:
len(X) == len(Y)

## Export 2 : same but min max norm / also export Y ref/probe as it was not made before

#### Data info
* Image min : 0
* Image max : 1491
* normalization = x / 1491

In [ ]:
def norm(x):
    return (x / 1491.0).astype(np.float32)

In [ ]:
# X = np.zeros((len(metadata_train), 101, 75, 40), dtype=np.float32) # too large
X = np.memmap('X_AllStepsR_f32_normMM.memmap', dtype=np.float32, mode="w+", shape=(len(metadata_train), 101, 75, 40))
Y = np.zeros((len(metadata_train), 3), dtype=np.uint8)
i = 0
flipped = 0

for row in metadata_train.itertuples(index=True):
    # not clean as we reload npz each time, but only run once...
    sample_path = datapath_train / f'{row.ParticipantID:03}' / row.Footwear / row.Speed / 'pipeline_1.npz'
    sample = np.load(sample_path)[f'{str(row.FootstepID)}'].astype(np.float32)

    # todo: if left do a flip
    if row.Side == 'Left':
        flipped += 1
        sample = sample[:, :, ::-1]
    
    X[i] = norm(sample)
    Y[i] = [row.ParticipantID, label_map_fw.get(row.Footwear), label_map_sp.get(row.Speed)]
    i += 1

X.flush()
# np.save('Y_AllStepsR.npy', Y)

In [ ]:
X = np.memmap("X_AllStepsR_f32_normMM.memmap", dtype=np.float32, mode="r", shape=(len(metadata_train), 101, 75, 40))

In [ ]:
len(X) == len(Y)

Same for references

In [ ]:
metadata_ref = pd.read_csv(datapath_ref / 'reference_metadata.csv',index_col = 'Index') 
footsteps_ref = np.load(datapath_ref / 'reference_data.npz')

In [ ]:
np.save('YRef_AllStepsR', np.array(metadata_ref).astype(np.uint8))

In [ ]:
X = np.zeros((len(metadata_ref)*2, 101, 75, 40), dtype=np.float32)

for i in range(len(footsteps_ref)):
    left = footsteps_ref[f'{i}'][0]
    left = left[:, :, ::-1]
    right = footsteps_ref[f'{i}'][1]
    
    left  = norm(left)
    right = norm(right)

    X[2*i] = left
    X[2*i+1] = right

In [ ]:
np.save('XRef_AllStepsR_f32_normMM.npy', X)

And for probe

In [ ]:
metadata_probe = pd.read_csv(datapath_probe / 'probe_metadata.csv',index_col = 'Index') 
footsteps_probe = np.load(datapath_probe / 'probe_data.npz')

In [ ]:
np.save('YProbe_AllStepsR', np.array(metadata_probe).astype(np.uint8))

In [ ]:
X = np.zeros((len(metadata_probe)*2, 101, 75, 40), dtype=np.float32)

for i in range(len(footsteps_probe)):
    left = footsteps_probe[f'{i}'][0]
    left = left[:, :, ::-1]
    right = footsteps_probe[f'{i}'][1]
    
    left  = norm(left)
    right = norm(right)

    X[2*i] = left
    X[2*i+1] = right

In [ ]:
np.save('XProbe_AllStepsR_f32_normMM.npy', X)

## Export 1 : extract all individual footstep

#### Data info
* Image min  : 0
* Image max  : 1491
* Image mean : 4.94
* Image std  : 24

#### My normalization idea : log -> norm
* log1p(max) = 7.3
* log1p(mean + std) = 3.5
* normalization = log1p(x) / 7

In [ ]:
# X = np.zeros((len(metadata_train), 101, 75, 40), dtype=np.float32) # too large
X = np.memmap('X_AllStepsR_f32_norm.memmap', dtype=np.float32, mode="w+", shape=(len(metadata_train), 101, 75, 40))
Y = np.zeros((len(metadata_train), 3), dtype=np.uint8)
i = 0
flipped = 0

for row in metadata_train.itertuples(index=True):
    # not clean as we reload npz each time, but only run once...
    sample_path = datapath_train / f'{row.ParticipantID:03}' / row.Footwear / row.Speed / 'pipeline_1.npz'
    sample = np.load(sample_path)[f'{str(row.FootstepID)}'].astype(np.float32)

    # todo: if left do a flip
    if row.Side == 'Left':
        flipped += 1
        sample = sample[:, :, ::-1]
    
    X[i] = np.log1p(sample) / 7.0
    Y[i] = [row.ParticipantID, label_map_fw.get(row.Footwear), label_map_sp.get(row.Speed)]
    i += 1

X.flush()
np.save('Y_AllStepsR.npy', Y)

In [ ]:
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(len(metadata_train), 101, 75, 40))

In [ ]:
len(X) == len(Y)

Same for references

In [ ]:
metadata_ref = pd.read_csv(datapath_ref / 'reference_metadata.csv',index_col = 'Index') 
footsteps_ref = np.load(datapath_ref / 'reference_data.npz')

In [ ]:
X = np.zeros((len(metadata_ref)*2, 101, 75, 40), dtype=np.float32)

for i in range(len(footsteps_ref)):
    left = footsteps_ref[f'{i}'][0]
    left = left[:, :, ::-1]
    right = footsteps_ref[f'{i}'][1]
    
    left  = np.log1p(left)  / 7.0
    right = np.log1p(right) / 7.0

    X[2*i] = left
    X[2*i+1] = right

In [ ]:
np.save('XRef_AllStepsR_f32_norm.npy', X)

And for probe

In [ ]:
metadata_probe = pd.read_csv(datapath_probe / 'probe_metadata.csv',index_col = 'Index') 
footsteps_probe = np.load(datapath_probe / 'probe_data.npz')

In [ ]:
X = np.zeros((len(metadata_probe)*2, 101, 75, 40), dtype=np.float32)

for i in range(len(footsteps_probe)):
    left = footsteps_probe[f'{i}'][0]
    left = left[:, :, ::-1]
    right = footsteps_probe[f'{i}'][1]
    
    left  = np.log1p(left)  / 7.0
    right = np.log1p(right) / 7.0

    X[2*i] = left
    X[2*i+1] = right

In [ ]:
np.save('XProbe_AllStepsR_f32_norm.npy', X)